# Branch And Bound 

## Import Library 

In [23]:
import math
import time
import heapq
import numpy as np
from scipy.optimize import linprog
from copy import deepcopy

## Read Data

In [24]:
def read_solomon_file(filepath: str) -> dict:
    with open(filepath, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    # Dòng chứa số xe và sức chứa
    n_vehicles = capacity = None
    data_rows  = []

    i = 0
    while i < len(lines):
        parts = lines[i].split()
        # Tìm dòng "25  200" (vehicle number + capacity)
        if (n_vehicles is None and len(parts) == 2
                and parts[0].isdigit() and parts[1].isdigit()):
            n_vehicles = int(parts[0])
            capacity   = int(parts[1])
        # Dòng dữ liệu khách hàng: 7 số
        elif len(parts) == 7 and parts[0].isdigit():
            data_rows.append([int(v) for v in parts])
        i += 1

    customers = [
        {
            "id":      r[0],
            "x":       r[1],
            "y":       r[2],
            "demand":  r[3], # nhu cau
            "ready":   r[4],
            "due":     r[5],
            "service": r[6],
        }
        for r in data_rows
    ]

    return {
        "n_vehicles": n_vehicles,
        "capacity":   capacity,
        "customers":  customers,   # index 0 = depot
    }

## Build Data

In [25]:
def build_problem(raw: dict, n_customers: int) -> dict:
    """
    Xây dựng dict bài toán VRPTW từ raw data Solomon.
    
    Trả về dict gồm:
        nodes, depot, customers, demand, ready, due, service,
        dist, capacity, n_vehicles, arcs, n, M_big
    """
    rows  = raw["customers"][: n_customers + 1]   # depot + n_customers
    nodes = [r["id"] for r in rows]

    x_coord = {r["id"]: r["x"]       for r in rows}
    y_coord = {r["id"]: r["y"]       for r in rows}
    demand  = {r["id"]: r["demand"]  for r in rows}
    ready   = {r["id"]: r["ready"]   for r in rows}
    due     = {r["id"]: r["due"]     for r in rows}
    service = {r["id"]: r["service"] for r in rows}

    # c_ij = t_ij = khoảng cách Euclidean (tốc độ = 1)
    dist = {
        (i, j): math.hypot(x_coord[i] - x_coord[j], y_coord[i] - y_coord[j])
        for i in nodes for j in nodes if i != j
    }

    arcs = [(i, j) for i in nodes for j in nodes if i != j]
    n    = len(nodes)   # tổng số nút (depot + khách hàng)

    # Big-M theo công thức: M_ij = b_i + t_ij + s_service_i - a_j
    M_time = {
        (i, j): max(0.0, due[i] + service[i] + dist[(i, j)] - ready[j])
        for (i, j) in arcs
    }

    return {
        "nodes":      nodes,
        "depot":      nodes[0],
        "customers":  nodes[1:],
        "demand":     demand,
        "ready":      ready,
        "due":        due,
        "service":    service,
        "dist":       dist,
        "capacity":   raw["capacity"],
        "n_vehicles": raw["n_vehicles"],
        "arcs":       arcs,
        "n":          n,
        "M_time":     M_time,
    }


## Giải LP Relataxion 

In [26]:
def solve_lp_relaxation(problem: dict,
                         fix_to_zero: set,
                         fix_to_one:  set) -> dict:
    """
    Giải LP Relaxation của mô hình VRPTW tại một nút B&B.

    Mô hình LP (Two-index formulation chuẩn):
        Biến: x[a] cho mỗi cung a=(i,j), s[i] thời gian bắt đầu phục vụ, u[i] tải tích lũy
        Hàm mục tiêu: min  sum_a c_a * x_a
        Ràng buộc:
            (1) sum_{j} x_{ij} = 1                    forall i in customers    (mỗi KH ra đúng 1 lần)
            (2) sum_{j} x_{hj} - sum_{i} x_{ih} = 0  forall h in customers    (bảo toàn luồng)
            (3a) sum_{j in C} x_{0j} <= m                                      (số xe xuất phát tối đa)
            (3b) sum_{i in C} x_{i0} <= m                                      (số xe về tối đa — FIX mới)
            (4) a_i <= s_i <= b_i                     forall i in nodes        (time window, encoded in bounds)
            (5) s_i + serv_i + t_ij - s_j <= M_ij*(1-x_ij)  forall (i,j) in arcs  (thứ tự thời gian)
                <=> s_i - s_j + M_ij * x_ij <= M_ij - t_ij - serv_i
                Đặc biệt khi j=depot: s_i + M_ij*x_ij <= M_ij + due[depot] - t_ij - serv_i
            (6) u_i + q_j - u_j <= Q*(1-x_ij)        forall (i,j), j != depot (sức chứa)
                <=> u_i - u_j + Q * x_ij <= Q - q_j
            (7) q_i <= u_i <= Q                       forall i in customers    (bounds)
            (8) 0 <= x_ij <= 1                                                 (nới lỏng nguyên)
            (+) x_ij = 0 nếu (i,j) in fix_to_zero
            (+) x_ij = 1 nếu (i,j) in fix_to_one

    Tham số
    -------
    problem      : dict từ build_problem()
    fix_to_zero  : tập cung (i,j) bị ép x_ij = 0 (nhánh trái)
    fix_to_one   : tập cung (i,j) bị ép x_ij = 1 (nhánh phải)

    Trả về
    ------
    dict: status ('optimal'/'infeasible'), obj, x_val, s_val, u_val
    """
    nodes      = problem["nodes"]
    customers  = problem["customers"]
    depot      = problem["depot"]
    arcs       = problem["arcs"]
    dist       = problem["dist"]
    ready      = problem["ready"]
    due        = problem["due"]
    demand     = problem["demand"]
    service    = problem["service"]
    capacity   = problem["capacity"]
    n_vehicles = problem["n_vehicles"]
    M_time     = problem["M_time"]
    n          = problem["n"]          # số nút
    na         = len(arcs)             # số cung

    # -------------------------------------------------------------------
    # Lập index:
    #   x[a]  : cột 0..na-1          (na biến)
    #   s[i]  : cột na..na+n-1       (n biến)
    #   u[i]  : cột na+n..na+2n-1   (n biến)
    # -------------------------------------------------------------------
    arc_idx  = {a: k    for k, a in enumerate(arcs)}
    node_idx = {v: k    for k, v in enumerate(nodes)}
    n_vars   = na + n + n

    def ix(a):  return arc_idx[a]           # index của x_a
    def is_(v): return na + node_idx[v]     # index của s_v
    def iu(v):  return na + n + node_idx[v] # index của u_v

    # -------------------------------------------------------------------
    # Hàm mục tiêu: min sum c_ij * x_ij   (s, u có hệ số 0)
    # -------------------------------------------------------------------
    c_obj = np.zeros(n_vars)
    for a in arcs:
        c_obj[ix(a)] = dist[a]

    # -------------------------------------------------------------------
    # Bounds
    # -------------------------------------------------------------------
    bounds = [None] * n_vars

    # x_ij in [0, 1]
    for a in arcs:
        lo, hi = 0.0, 1.0
        if a in fix_to_zero: lo = hi = 0.0
        if a in fix_to_one:  lo = hi = 1.0
        bounds[ix(a)] = (lo, hi)

    # s_i in [a_i, b_i]
    for v in nodes:
        bounds[is_(v)] = (float(ready[v]), float(due[v]))

    # u_depot = 0 (depot không mang hàng)
    bounds[iu(depot)] = (0.0, 0.0)
    # u_i in [q_i, Q] cho khách hàng
    for v in customers:
        bounds[iu(v)] = (float(demand[v]), float(capacity))

    # -------------------------------------------------------------------
    # Ràng buộc bằng (equality): A_eq @ z = b_eq
    # (1) sum_{j: j!=i} x_{ij} = 1  forall i in customers
    # (2) sum_{j} x_{hj} - sum_{i} x_{ih} = 0  forall h in customers
    # -------------------------------------------------------------------
    A_eq_rows, b_eq = [], []

    # Ràng buộc (1): mỗi khách hàng xuất đi đúng 1 lần
    for i in customers:
        row = np.zeros(n_vars)
        for j in nodes:
            if j != i and (i, j) in arc_idx:
                row[ix((i, j))] = 1.0
        A_eq_rows.append(row)
        b_eq.append(1.0)

    # Ràng buộc (2): bảo toàn luồng tại mỗi khách hàng (out - in = 0)
    for h in customers:
        row = np.zeros(n_vars)
        for j in nodes:
            if j != h and (h, j) in arc_idx:
                row[ix((h, j))] += 1.0   # ra khỏi h
        for i in nodes:
            if i != h and (i, h) in arc_idx:
                row[ix((i, h))] -= 1.0   # vào h
        A_eq_rows.append(row)
        b_eq.append(0.0)

    A_eq = np.array(A_eq_rows) if A_eq_rows else None
    b_eq = np.array(b_eq)

    # -------------------------------------------------------------------
    # Ràng buộc bất đẳng thức (upper bound): A_ub @ z <= b_ub
    # (3a) sum_{j in C} x_{0j} <= m   (số xe xuất phát)
    # (3b) sum_{i in C} x_{i0} <= m   (số xe về depot)   ← FIX: thêm mới
    # (5)  s_i + serv_i + t_ij - s_j <= M_ij*(1 - x_ij)
    #      => s_i - s_j + M_ij * x_ij <= M_ij - t_ij - serv_i
    #      Trường hợp j=depot: s_j không bị ràng buộc nhất quán với route,
    #      nên viết: s_i + M_ij*x_ij <= M_ij + due[depot] - t_ij - serv_i
    # (6)  u_i - u_j + Q * x_ij <= Q - q_j  (bỏ qua j=depot vì u_depot=0 là giả)
    # -------------------------------------------------------------------
    A_ub_rows, b_ub = [], []

    # Ràng buộc (3a): số xe xuất phát từ depot tối đa
    row = np.zeros(n_vars)
    for j in customers:
        if (depot, j) in arc_idx:
            row[ix((depot, j))] = 1.0
    A_ub_rows.append(row)
    b_ub.append(float(n_vehicles))

    # FIX (3b): số xe về depot tối đa — đảm bảo đối xứng với (3a)
    row = np.zeros(n_vars)
    for i in customers:
        if (i, depot) in arc_idx:
            row[ix((i, depot))] = 1.0
    A_ub_rows.append(row)
    b_ub.append(float(n_vehicles))

    # Ràng buộc (5): thứ tự thời gian
    # Công thức chuẩn: s_i + serv_i + t_ij <= s_j + M_ij*(1 - x_ij)
    # => s_i - s_j + M_ij * x_ij <= M_ij - t_ij - serv_i
    # Trường hợp đặc biệt j == depot:
    #   Không dùng s_depot trong ràng buộc (s_depot chỉ đại diện thời gian
    #   xuất phát lần đầu, không phải lần về). Thay bằng:
    #   s_i + serv_i + t_i0 <= due[depot]  khi x_i0 = 1
    #   => s_i + M_ij * x_ij <= M_ij + due[depot] - t_ij - serv_i   ← FIX: thêm serv_i
    for (i, j) in arcs:
        M_ij   = M_time[(i, j)]
        t_ij   = dist[(i, j)]
        serv_i = float(service[i])   # FIX: luôn dùng service[i], kể cả depot (=0)
        row    = np.zeros(n_vars)

        if j == depot:
            # FIX: ràng buộc đúng — s_i + serv_i + t_ij <= due[depot] khi x_ij=1
            # => s_i + M_ij * x_ij <= M_ij + due[depot] - t_ij - serv_i
            row[is_(i)]     =  1.0
            # Không đưa s_depot vào — depot return không cần thứ tự thời gian
            row[ix((i, j))] =  M_ij
            A_ub_rows.append(row)
            b_ub.append(M_ij + float(due[depot]) - t_ij - serv_i)

        else:
            # FIX: thống nhất dùng serv_i cho cả i=depot và i=customer
            # s_i - s_j + M_ij * x_ij <= M_ij - t_ij - serv_i
            row[is_(i)]     =  1.0
            row[is_(j)]     = -1.0
            row[ix((i, j))] =  M_ij
            A_ub_rows.append(row)
            b_ub.append(M_ij - t_ij - serv_i)

    # Ràng buộc (6): sức chứa
    # u_i + q_j - u_j <= Q * (1 - x_ij)
    # => u_i - u_j + Q * x_ij <= Q - q_j
    # Bỏ qua j=depot: u_depot=0 là giả (depot không có nhu cầu và
    # tải được reset khi xe về depot — tải chỉ tăng trong 1 route).
    Q = float(capacity)
    for (i, j) in arcs:
        if j == depot:
            continue  # skip: tải không tích lũy qua depot
        q_j = float(demand[j])
        row = np.zeros(n_vars)
        row[iu(i)]      =  1.0
        row[iu(j)]      = -1.0
        row[ix((i, j))] =  Q
        A_ub_rows.append(row)
        b_ub.append(Q - q_j)

    A_ub = np.array(A_ub_rows) if A_ub_rows else None
    b_ub = np.array(b_ub)

    # -------------------------------------------------------------------
    # Gọi HiGHS solver qua scipy.optimize.linprog
    # -------------------------------------------------------------------
    result = linprog(
        c_obj,
        A_ub=A_ub, b_ub=b_ub,
        A_eq=A_eq, b_eq=b_eq,
        bounds=bounds,
        method="highs",
        options={"disp": False, "time_limit": 1800},
    )

    if result.status != 0:   # không tìm được nghiệm khả thi
        return {"status": "infeasible"}

    # Trích xuất giá trị nghiệm
    z       = result.x
    x_val   = {a: z[ix(a)]   for a in arcs}
    s_val   = {v: z[is_(v)]  for v in nodes}
    u_val   = {v: z[iu(v)]   for v in nodes}

    return {
        "status": "optimal",
        "obj":    result.fun,
        "x_val":  x_val,
        "s_val":  s_val,
        "u_val":  u_val,
    }


## Chọn biến phân nhánh

In [27]:
def select_branching_variable(x_val: dict,
                               fix_to_zero: set,
                               fix_to_one:  set,
                               tol: float = 1e-5):
    """
    Chọn biến x_{ij} phân số để phân nhánh.

    Chiến lược: Most Fractional — chọn biến có giá trị gần 0.5 nhất,
    tức là min |x_ij - 0.5|.

    Tham số
    -------
    x_val       : dict {(i,j): float} nghiệm LP
    fix_to_zero : tập cung đã cố định = 0 (bỏ qua)
    fix_to_one  : tập cung đã cố định = 1 (bỏ qua)
    tol         : ngưỡng phân số

    Trả về
    ------
    (i, j) hoặc None nếu tất cả biến đã nguyên
    """
    best_arc  = None
    best_dist = math.inf

    for arc, val in x_val.items():
        if arc in fix_to_zero or arc in fix_to_one:
            continue
        # Biến phân số: không nguyên trong phạm vi [tol, 1-tol]
        if tol < val < 1.0 - tol:
            d = abs(val - 0.5)
            if d < best_dist:
                best_dist = d
                best_arc  = arc

    return best_arc

## Chích xuất các nghiệm nguyên 

In [28]:
def extract_routes(x_val: dict, problem: dict, tol: float = 1e-5) -> list:
    """
    Từ nghiệm x_val nguyên, trích xuất danh sách tuyến đường.

    Tham số
    -------
    x_val   : dict {(i,j): float}, các x_ij ~1 là cung được chọn
    problem : dict bài toán

    Trả về
    ------
    list of dict {customers, cost}
    """
    depot = problem["depot"]
    dist  = problem["dist"]

    # Xây dựng đồ thị successor từ x_val
    succ = {}
    for (i, j), v in x_val.items():
        if v > 1.0 - tol:   # x_ij ≈ 1
            succ[i] = j

    routes = []

    # FIX: lấy tất cả start nodes (khách hàng ngay sau depot)
    # Không dùng set visited_depots vì mỗi start node là duy nhất trong
    # nghiệm nguyên hợp lệ (mỗi cung depot->j chỉ xuất hiện 1 lần).
    start_nodes = [j for (i, j), v in x_val.items()
                   if i == depot and v > 1.0 - tol]

    for start in start_nodes:
        route_customers = []
        cur  = start
        cost = dist[(depot, start)]
        seen = set()

        while cur != depot:
            if cur in seen:      # phòng vòng lặp bất thường
                break
            seen.add(cur)
            route_customers.append(cur)
            nxt = succ.get(cur)
            if nxt is None:
                break
            cost += dist[(cur, nxt)]
            cur   = nxt

        if route_customers:
            routes.append({"customers": route_customers, "cost": cost})

    return routes


## Branch and Bound 

In [29]:
def branch_and_bound(problem: dict, time_limit: float = 120.0) -> dict:
    """
    Thuật toán Branch and Bound (Best-First Search) cho VRPTW.

    Mỗi nút trên cây B&B lưu:
        - fix_to_zero : set các cung (i,j) bị ép x_ij = 0
        - fix_to_one  : set các cung (i,j) bị ép x_ij = 1
        - lb          : cận dưới (giá trị LP Relaxation của nút này)

    Chiến lược phân nhánh:
        - Chọn biến x_ij phân số gần 0.5 nhất (Most Fractional)
        - Tạo nhánh trái  : x_ij = 0 (cấm cung i→j)
        - Tạo nhánh phải  : x_ij = 1 (bắt buộc cung i→j)

    Tham số
    -------
    problem    : dict từ build_problem()
    time_limit : giới hạn thời gian (giây)

    Trả về
    ------
    dict gồm best_cost, best_routes, nodes_explored, nodes_pruned,
              nodes_pruned_infeasible, nodes_pruned_bound,
              nodes_pruned_optimal, elapsed
    """
    start_time = time.time()

    # -------------------------------------------------------------------
    # Bước 1 — Khởi tạo
    # best_cost = UB = +inf (cận trên, nghiệm nguyên tốt nhất hiện tại)
    # -------------------------------------------------------------------
    best_cost   = math.inf
    best_routes = None

    nodes_explored         = 0
    nodes_pruned_infeasible = 0
    nodes_pruned_bound      = 0
    nodes_pruned_optimal    = 0

    # Heap (priority queue): phần tử = (lb, node_id, fix_zero, fix_one)
    # Best-First: pop nút có lb nhỏ nhất trước
    node_counter = 0
    root_node    = (0.0, node_counter, frozenset(), frozenset())  # lb=0 cho root
    heap         = [root_node]
    heapq.heapify(heap)

    while heap:
        # -------------------------------------------------------------------
        # Bước 2 — Kiểm tra time limit và lấy nút
        # -------------------------------------------------------------------
        if time.time() - start_time > time_limit:
            print(f"    [!] Hết time_limit = {time_limit}s — trả về nghiệm tốt nhất hiện tại.")
            break

        lb, _, fix_zero, fix_one = heapq.heappop(heap)
        nodes_explored += 1

        # -------------------------------------------------------------------
        # Bước 3 — Bounding: giải LP Relaxation tại nút hiện tại
        # -------------------------------------------------------------------
        lp_result = solve_lp_relaxation(problem,
                                         fix_to_zero=set(fix_zero),
                                         fix_to_one =set(fix_one))

        # -------------------------------------------------------------------
        # Bước 4 — Pruning
        # -------------------------------------------------------------------

        # Prune by Infeasibility: LP vô nghiệm
        if lp_result["status"] == "infeasible":
            nodes_pruned_infeasible += 1
            continue

        Z_LP = lp_result["obj"]

        # Prune by Bound: cận dưới >= nghiệm tốt nhất hiện tại
        if Z_LP >= best_cost - 1e-8:
            nodes_pruned_bound += 1
            continue

        # Kiểm tra nghiệm nguyên: tất cả x_ij ∈ {0, 1}
        x_val       = lp_result["x_val"]
        branch_var  = select_branching_variable(x_val, set(fix_zero), set(fix_one))

        # Prune by Optimality: không có biến phân số → nghiệm nguyên
        if branch_var is None:
            if Z_LP < best_cost:
                best_cost   = Z_LP
                best_routes = extract_routes(x_val, problem)
            nodes_pruned_optimal += 1
            continue

        # -------------------------------------------------------------------
        # Bước 5 — Branching: tạo hai nhánh từ biến x_{ij} phân số
        # -------------------------------------------------------------------
        i_b, j_b = branch_var

        # Nhánh trái: x_{ij} = 0 (cấm cung i→j)
        left_zero = fix_zero | frozenset([(i_b, j_b)])
        left_one  = fix_one
        node_counter += 1
        heapq.heappush(heap, (Z_LP, node_counter, left_zero, left_one))

        # Nhánh phải: x_{ij} = 1 (bắt buộc cung i→j)
        right_zero = fix_zero
        right_one  = fix_one | frozenset([(i_b, j_b)])
        node_counter += 1
        heapq.heappush(heap, (Z_LP, node_counter, right_zero, right_one))

    elapsed = time.time() - start_time

    return {
        "best_cost":              best_cost,
        "best_routes":            best_routes,
        "nodes_explored":         nodes_explored,
        "nodes_pruned_infeasible": nodes_pruned_infeasible,
        "nodes_pruned_bound":     nodes_pruned_bound,
        "nodes_pruned_optimal":   nodes_pruned_optimal,
        "elapsed":                elapsed,
        "n_vehicles_used":        len(best_routes) if best_routes else 0,
    }

## Print Result 

In [30]:
def print_result(n: int, result: dict):
    """
    In chi tiết kết quả của một instance.

    Tham số
    -------
    n      : số khách hàng
    result : dict từ branch_and_bound()
    """
    print(f"\n{'='*65}")
    print(f"  Instance C101_n{n:02d}  |  n = {n} khách hàng")
    print(f"{'='*65}")

    if result["best_routes"] is None:
        print("  Không tìm được nghiệm khả thi trong giới hạn thời gian.")
        return

    print(f"  Tổng chi phí (Z*)          : {result['best_cost']:.4f}")
    print(f"  Số xe sử dụng              : {result['n_vehicles_used']}")
    print(f"  Số nút B&B đã duyệt        : {result['nodes_explored']}")
    print(f"  Cắt tỉa (infeasible)       : {result['nodes_pruned_infeasible']}")
    print(f"  Cắt tỉa (by bound)         : {result['nodes_pruned_bound']}")
    print(f"  Cắt tỉa (optimality/int)   : {result['nodes_pruned_optimal']}")
    print(f"  Thời gian chạy             : {result['elapsed']:.4f}s")
    print(f"\n  Chi tiết các tuyến:")
    for idx, route in enumerate(result["best_routes"], 1):
        cust_str = " -> ".join(str(c) for c in route["customers"])
        print(f"    Xe {idx:2d}: 0 -> {cust_str} -> 0"
              f"  (cost = {route['cost']:.4f})")

## Visualization

In [31]:
import matplotlib
import matplotlib.pyplot as plt
import os

matplotlib.use("Agg")


def solVis(problem: dict, routes: list, sol_time: float, opt_cost: float,
           dataset_name: str, save_dir: str = ".", show: bool = False):
    """
    Visualize kết quả VRPTW.

    Tham số
    -------
    problem      : dict từ build_problem_vis() — chứa _raw_rows với toạ độ
    routes       : list of dict {customers, cost} từ extract_routes()
    sol_time     : thời gian giải (giây)
    opt_cost     : tổng chi phí tối ưu
    dataset_name : tên dataset (dùng trong tiêu đề và tên file lưu)
    save_dir     : thư mục lưu ảnh PNG (mặc định thư mục hiện tại)
    show         : True → hiển thị inline (Jupyter / Colab)
    """
    if "_raw_rows" not in problem:
        raise ValueError(
            "problem dict thiếu _raw_rows. Hãy dùng build_problem_vis() "
            "thay vì build_problem() để lưu toạ độ."
        )

    x_coord  = {r["id"]: r["x"] for r in problem["_raw_rows"]}
    y_coord  = {r["id"]: r["y"] for r in problem["_raw_rows"]}
    depot    = problem["depot"]
    customers = problem["customers"]

    depot_x, depot_y = x_coord[depot], y_coord[depot]

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_title(
        f"B&B Solution for VRPTW — Dataset {dataset_name}",
        fontsize=15, fontweight="bold", pad=14
    )

    # Depot
    ax.scatter(depot_x, depot_y, color="black", s=140, zorder=6, label="Depot")
    ax.annotate("Depot", (depot_x, depot_y),
                textcoords="offset points", xytext=(6, 6),
                fontsize=10, color="black", fontweight="bold")

    # Customers
    cx = [x_coord[c] for c in customers]
    cy = [y_coord[c] for c in customers]
    ax.scatter(cx, cy, color="steelblue", s=60, zorder=5, label="Customers")
    for c in customers:
        ax.annotate(str(c), (x_coord[c], y_coord[c]),
                    textcoords="offset points", xytext=(4, 4),
                    fontsize=8, color="#333333")

    # Routes
    cmap = plt.cm.tab20
    for idx, route in enumerate(routes):
        color = cmap(idx % cmap.N)
        path  = [depot] + route["customers"] + [depot]
        rx = [x_coord[v] for v in path]
        ry = [y_coord[v] for v in path]

        ax.plot(rx, ry, color=color, linewidth=1.8,
                label=f"Route {idx+1}  (cost={route['cost']:.1f})", zorder=2)

        # Mũi tên ở điểm giữa mỗi đoạn
        for k in range(len(path) - 1):
            sx, sy = rx[k], ry[k]
            ex, ey = rx[k+1], ry[k+1]
            mx, my = (sx + ex) / 2, (sy + ey) / 2
            ax.annotate("", xy=(mx, my), xytext=(sx, sy),
                        arrowprops=dict(arrowstyle="->", color=color, lw=1.4),
                        zorder=3)

    ax.grid(True, linestyle="--", alpha=0.5)
    ax.set_xlabel("X Coordinate", fontsize=12)
    ax.set_ylabel("Y Coordinate", fontsize=12)
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1),
              borderaxespad=0, fontsize=9)

    info = (
        f"Optimal Cost : {opt_cost:.2f}\n"
        f"Routes used  : {len(routes)}\n"
        f"Solve time   : {sol_time:.2f}s"
    )
    ax.text(0.02, 0.02, info, transform=ax.transAxes, fontsize=9,
            va="bottom", ha="left",
            bbox=dict(facecolor="white", edgecolor="#aaaaaa",
                      boxstyle="round,pad=0.5", alpha=0.9))

    plt.tight_layout()

    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"VRPTW_BnB_Sol_{dataset_name}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"[solVis] Đã lưu hình: {save_path}")

    if show:
        plt.show()
    plt.close()


def build_problem_vis(raw: dict, n_customers: int) -> dict:
    """
    Giống build_problem() nhưng lưu thêm _raw_rows vào dict
    để solVis có thể đọc toạ độ x, y của từng node.
    """
    problem = build_problem(raw, n_customers)
    rows    = raw["customers"][: n_customers + 1]   # depot + n_customers
    problem["_raw_rows"] = rows
    return problem


## Run 

In [32]:
def run_experiments(filepath: str = "C101.txt", save_dir: str = ".", show_plot: bool = False):
    """
    Chạy B&B cho C101 với n = 5, 10, 15, 25 và visualize từng kết quả.

    Tham số
    -------
    filepath  : đường dẫn tới file C101.txt (hoặc Solomon format khác)
    save_dir  : thư mục lưu ảnh PNG (mặc định thư mục hiện tại)
    show_plot : True → hiển thị inline trong Jupyter/Colab
    """
    print(f"[*] Đọc dữ liệu từ: {filepath}")
    raw = read_solomon_file(filepath)
    print(f"    Số xe: {raw['n_vehicles']}  |  Sức chứa: {raw['capacity']}")

    # Tên dataset lấy từ tên file (bỏ phần mở rộng)
    dataset_base = os.path.splitext(os.path.basename(filepath))[0]

    configs = [
        {"n":  5, "time_limit": 180000000},
        #{"n": 10, "time_limit":  18000000},
        #{"n": 15, "time_limit":  18000000},
        #{"n": 25, "time_limit":  18000000},
    ]

    summary = []

    for cfg in configs:
        n  = cfg["n"]
        tl = cfg["time_limit"]

        print(f"\n[*] Giải {dataset_base}_n{n:02d}  (time_limit={tl}s) ...")
        # Dùng build_problem_vis để lưu toạ độ cho visualization
        problem = build_problem_vis(raw, n)
        result  = branch_and_bound(problem, time_limit=tl)
        print_result(n, result)

        # ── Visualize nếu tìm được nghiệm ──
        if result["best_routes"]:
            dataset_name = f"{dataset_base}_n{n:02d}"
            solVis(
                problem      = problem,
                routes       = result["best_routes"],
                sol_time     = result["elapsed"],
                opt_cost     = result["best_cost"],
                dataset_name = dataset_name,
                save_dir     = save_dir,
                show         = show_plot,
            )

        summary.append({
            "n":        n,
            "cost":     f"{result['best_cost']:.4f}" if result["best_routes"] else "N/A",
            "vehicles": result["n_vehicles_used"],
            "nodes":    result["nodes_explored"],
            "time_s":   f"{result['elapsed']:.4f}",
        })

    # Bảng tổng hợp
    print(f"\n\n{'='*70}")
    print(f"  KẾT QUẢ THỰC NGHIỆM — Branch and Bound (MILP) — {dataset_base}")
    print(f"{'='*70}")
    header = (f"  {'Instance':<14} {'Z* (tối ưu)':>14} "
              f"{'Số xe':>7} {'Số nút B&B':>12} {'T.gian (s)':>12}")
    print(header)
    print(f"  {'-'*14} {'-'*14} {'-'*7} {'-'*12} {'-'*12}")
    for s in summary:
        inst = f"{dataset_base}_n{s['n']:02d}"
        print(f"  {inst:<14} {s['cost']:>14} {s['vehicles']:>7}"
              f" {s['nodes']:>12} {s['time_s']:>12}")
    print(f"{'='*70}\n")


In [33]:
filepath = "F:\hoc_ki_2_nam_2025_2026\machine_learning\exercise\daa-project\data\C101.txt"
run_experiments(filepath)

[*] Đọc dữ liệu từ: F:\hoc_ki_2_nam_2025_2026\machine_learning\exercise\daa-project\data\C101.txt
    Số xe: 25  |  Sức chứa: 200

[*] Giải C101_n05  (time_limit=180000000s) ...

  Instance C101_n05  |  n = 5 khách hàng
  Tổng chi phí (Z*)          : 42.4198
  Số xe sử dụng              : 1
  Số nút B&B đã duyệt        : 9
  Cắt tỉa (infeasible)       : 2
  Cắt tỉa (by bound)         : 2
  Cắt tỉa (optimality/int)   : 1
  Thời gian chạy             : 0.0243s

  Chi tiết các tuyến:
    Xe  1: 0 -> 5 -> 3 -> 4 -> 2 -> 1 -> 0  (cost = 42.4198)
[solVis] Đã lưu hình: .\VRPTW_BnB_Sol_C101_n05.png


  KẾT QUẢ THỰC NGHIỆM — Branch and Bound (MILP) — C101
  Instance          Z* (tối ưu)   Số xe   Số nút B&B   T.gian (s)
  -------------- -------------- ------- ------------ ------------
  C101_n05              42.4198       1            9       0.0243

